# 11 — Incorporación retrospectiva de TEST1 (HAI22.04)

**No reentrena ni recalibra.** Lee los modelos `.joblib` y los umbrales originales; genera las predicciones, las detecciones y sus métricas para `test1.csv`. Mantiene intactas las salidas de TEST2–TEST4 y almacena los nuevos archivos en `resultados_hai_arboles/analisis/test1`.

**Advertencia metodológica:** TEST1 se informa como evaluación retrospectiva separada si precede a TRAIN1. Las etiquetas `Attack` se emplean únicamente para el contraste posterior, nunca para decidir el umbral. Se conservan las 24 señales y los mismos cuatro rezagos OSA de los árboles.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from joblib import load

# Ajustar las rutas SOLO si el proyecto está instalado en otra ubicación.
PROJECT = Path(r"C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive")
DATA_DIR = PROJECT / 'hai-22.04'
ROOT = PROJECT / 'resultados_hai_arboles'
OUT = ROOT / 'analisis' / 'test1'
OUT.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.is_dir(), f'No existe el directorio de datos: {DATA_DIR}'
assert ROOT.is_dir(), f'No existe el directorio de resultados: {ROOT}'
assert (DATA_DIR / 'test1.csv').exists(), 'No existe test1.csv'
assert (ROOT / 'modelos').is_dir(), 'No existen los modelos entrenados; no se reentrenará.'
print('Datos:', DATA_DIR)
print('Modelos originales:', ROOT / 'modelos')
print('Salidas nuevas:', OUT)

## 1. Inventario temporal y verificación de TEST1

Se imprimen las fechas reales de TEST1 y TRAIN1 para comprobar, en lugar de asumir, su relación cronológica.

In [ ]:
def period(name):
    p = DATA_DIR / name
    x = pd.read_csv(p, usecols=['timestamp', 'Attack'] if name.startswith('test') else ['timestamp'])
    ts = pd.to_datetime(x.timestamp)
    return {'archivo':name,'filas':len(x),'inicio':ts.iloc[0], 'fin':ts.iloc[-1],
            'episodios':int(((x.Attack == 1) & (x.Attack.shift(fill_value=0) != 1)).sum()) if 'Attack' in x else None}
periods = pd.DataFrame([period('test1.csv'), period('train1.csv')])
display(periods)
periods.to_csv(OUT / '01_periodos_test1_train1.csv', index=False)
print('TEST1 anterior a TRAIN1:', periods.loc[0,'fin'] < periods.loc[1,'inicio'])

## 2. Carga y comprobación de artefactos ya entrenados

Los 20 modelos guardados corresponden a cuatro algoritmos por cinco configuraciones. **Si falta alguno, la ejecución se detiene**; nunca genera un modelo nuevo. Persistencia se calcula directamente desde las observaciones y no necesita `.joblib`.

In [ ]:
METHODS = ('DT','RF','ET','XGB')
CONFIGS = ('FULL','SIN_P1','SIN_P2','SIN_P3','SIN_P4')
TARGETS = ['P1_FT01','P1_FT02','P1_FT03','P1_LIT01','P1_PIT01','P1_PIT02','P1_TIT01','P1_TIT02','P1_TIT03',
           'P2_SIT01','P2_VIBTR01','P2_VIBTR02','P2_VIBTR03','P2_VIBTR04','P2_VT01',
           'P3_FIT01','P3_LIT01','P3_PIT01',
           'P4_HT_FD','P4_HT_PO','P4_ST_FD','P4_ST_PO','P4_ST_PT01','P4_ST_TT01']
assert len(TARGETS)==24
missing=[]
for model in METHODS:
    for conf in CONFIGS:
        if not (ROOT/'modelos'/f'{model}_{conf}.joblib').exists(): missing.append(f'{model}_{conf}.joblib')
for model,conf in [(m,c) for m in METHODS for c in CONFIGS] + [('PERSIST','FULL')]:
    if not (ROOT/'deteccion'/f'umbral_{model}_{conf}.json').exists():missing.append(f'umbral_{model}_{conf}.json')
if missing: raise FileNotFoundError('Faltan artefactos originales (no reentrenar automáticamente): '+', '.join(missing))
print('Artefactos originales comprobados: 20 modelos y 21 umbrales.')
test = pd.read_csv(DATA_DIR/'test1.csv')
assert all(c in test for c in TARGETS + ['timestamp','Attack'])
assert len(test)>=31
assert not test[TARGETS].isna().any().any(), 'Se detectaron valores faltantes en señales objetivo'
print('Registros originales:',len(test),'; evaluables OSA:',len(test)-30)

## 3. Funciones equivalentes al protocolo original

La indexación usa `idx = 29, ..., n-2`, entrada en `t` y objetivo en `t+1`; se generan `n-30` predicciones. Se aplica el transformador guardado y se reconstruyen las salidas con `ymean` y `ystd`. Los parámetros de calibración (`mu`, `std`, `umbral`) se leen del archivo JSON **sin recalcularlos**.

In [ ]:
def target_array(df):
    idx = np.arange(30,len(df), dtype=np.int64)
    return idx, df[TARGETS].iloc[idx].to_numpy(dtype=np.float32)

def predict_saved(obj, df):
    features = obj['features']; offsets = tuple(obj['offsets']); L=int(obj['window'])
    if offsets != (0,1,8,29) or L!=30: raise ValueError('La representación temporal guardada difiere del protocolo.')
    if list(obj['targets']) != TARGETS: raise ValueError('Orden de salidas diferente en el modelo.')
    a=df[features].to_numpy(dtype=np.float32,copy=False)
    t=np.arange(L-1,len(df)-1)
    raw=np.concatenate([a[t-o] for o in offsets], axis=1).astype(np.float32,copy=False)
    yhat=obj['model'].predict(obj['scaler'].transform(raw))*obj['ystd']+obj['ymean']
    return np.asarray(yhat,dtype=np.float32)

def load_threshold(method, conf):
    cfg=json.loads((ROOT/'deteccion'/f'umbral_{method}_{conf}.json').read_text(encoding='utf-8'))
    if list(cfg['variables'])!=TARGETS:raise ValueError('El orden de variables del umbral es incompatible')
    if abs(float(cfg['cuantil'])-0.995)>1e-10:raise ValueError('Se detectó otro cuantil original')
    mu=np.asarray(cfg['mu'],dtype=float);sd=np.asarray(cfg['std'],dtype=float)
    if mu.shape!=(24,) or sd.shape!=(24,) or np.any(sd<=0):raise ValueError('Estadísticas de calibración inválidas')
    return mu,sd,float(cfg['umbral'])

def series_intervals(flag):
    a=np.asarray(flag,dtype=bool)
    starts=np.flatnonzero(a & ~np.r_[False,a[:-1]])
    ends=np.flatnonzero(a & ~np.r_[a[1:],False])
    return list(zip(starts,ends))

def r2_vec(y,p):
    resid=y-p;ss=np.sum(resid.astype(float)**2,axis=0)
    base=np.sum((y.astype(float)-y.mean(axis=0))**2,axis=0)
    return np.where(base>0,1-ss/np.where(base==0,1,base),np.nan)


## 4. Predicción y detección de TEST1

Se crean los archivos comprimidos con el mismo esquema de columnas que el programa original, pero dentro de `analisis/test1`. Las etiquetas se copian al final para el contraste y no se emplean para predecir o calcular la puntuación.

In [ ]:
idx,y=target_array(test)
rows=[]; episodios=[]; comparaciones=[]
labels=test['Attack'].iloc[idx].to_numpy(dtype=int)
assert set(np.unique(labels)).issubset({0,1})
for method in METHODS+('PERSIST',):
    for conf in (CONFIGS if method!='PERSIST' else ('FULL',)):
        if method=='PERSIST':
            pred=test[TARGETS].iloc[idx-1].to_numpy(dtype=np.float32)
        else:
            obj=load(ROOT/'modelos'/f'{method}_{conf}.joblib')
            if obj.get('method')!=method or obj.get('config')!=conf:
                raise ValueError(f'Identidad de modelo incoherente: {method}_{conf}')
            pred=predict_saved(obj,test)
        if pred.shape != y.shape:raise ValueError(f'Dimensiones incompatibles: {method}_{conf}')
        err=y-pred
        mu,sd,tau=load_threshold(method,conf)
        score=np.sqrt(np.mean(((err.astype(float)-mu)/sd)**2,axis=1))
        alert=score>tau
        prefix=f'{method}_{conf}_test1'
        frame=pd.DataFrame({'archivo':'test1','indice':idx,'timestamp':test.timestamp.iloc[idx].to_numpy(),'Attack':labels})
        for j,target in enumerate(TARGETS):
            frame['real__'+target]=y[:,j]
            frame['pred__'+target]=pred[:,j]
            frame['error__'+target]=err[:,j]
        frame.to_csv(OUT/f'{prefix}_predicciones.csv.gz',index=False,compression='gzip')
        det=pd.DataFrame({'archivo':'test1','indice':idx,'timestamp':test.timestamp.iloc[idx].to_numpy(),
                          'score':score,'alerta':alert.astype(int),'Attack':labels})
        det.to_csv(OUT/f'{prefix}_deteccion.csv.gz',index=False,compression='gzip')
        for j,target in enumerate(TARGETS):
            e=err[:,j].astype(float)
            comparaciones.append({'metodo':method,'configuracion':conf,'archivo':'test1','variable':target,'componente':target[:2],
                'rmse':float(np.sqrt(np.mean(e*e))),'mae':float(np.mean(np.abs(e))), 'r2':float(r2_vec(y[:,[j]],pred[:,[j]])[0])})
        eps=series_intervals(labels==1)
        for k,(st,en) in enumerate(eps,1):
            covered=int(alert[st:en+1].sum())
            episodios.append({'metodo':method,'configuracion':conf,'archivo':'test1','episodio':k,
                'inicio_indice':int(idx[st]),'fin_indice':int(idx[en]),'longitud':int(en-st+1),
                'registros_senalados':covered,'cobertura':covered/(en-st+1),'detectado':int(covered>0)})
        rows.append({'metodo':method,'configuracion':conf,'archivo':'test1','n':len(idx),
            'n_alerta':int(alert.sum()),'pds':float(100*alert.mean()),'episodios':len(eps),
            'episodios_coincidentes':sum(alert[st:en+1].any() for st,en in eps),
            'intervalos_senalados':len(series_intervals(alert)),
            'r2_mediano':float(np.nanmedian(r2_vec(y,pred))),'umbral_original':tau})
        print(f'{prefix}: {len(idx):,} filas, {int(alert.sum()):,} señaladas, {sum(alert[st:en+1].any() for st,en in eps)}/{len(eps)} episodios')
summary=pd.DataFrame(rows)
summary.to_csv(OUT/'02_resumen_test1.csv',index=False)
pd.DataFrame(comparaciones).to_csv(OUT/'03_metricas_por_senal_test1.csv',index=False)
pd.DataFrame(episodios).to_csv(OUT/'04_episodios_test1.csv',index=False)
display(summary)
print('Resultados creados en',OUT)

## 5. Control de integridad y siguientes comparaciones

No se modificó ningún artefacto original. Las filas de `02_resumen_test1.csv` permiten incorporar los resultados por separado a tablas posteriores. **No reemplace automáticamente el total de 51 episodios**: compruebe primero que TEST1 contiene 7 episodios evaluables y que las fechas están correctamente representadas. La comparación conjunta de TEST1–TEST4 y la actualización del artículo se harán después de revisar estos resultados.

In [ ]:
assert len(summary)==21, f'Se esperaban 21 combinaciones, hay {len(summary)}'
assert (summary.n==len(test)-30).all()
assert (summary.episodios==summary.episodios.iloc[0]).all()
assert (summary.n_alerta>=0).all() and (summary.n_alerta<=summary.n).all()
print('Combinaciones comprobadas:',len(summary))
print('Episodios etiquetados evaluables TEST1:',int(summary.episodios.iloc[0]))
print('Se conservaron los modelos, los umbrales y las salidas anteriores.')